In [8]:
# ==============================================================================
# IMPORT REQUIRED LIBRARIES
# ==============================================================================

import os
import random
import numpy as np
import pandas as pd

from faker import Faker
from datetime import datetime


# ==============================================================================
# INITIALIZE RANDOM GENERATORS
# ==============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

fake = Faker("en_IN")
Faker.seed(SEED)

print("Libraries imported successfully.")
print(f"Random seed: {SEED}")

Libraries imported successfully.
Random seed: 42


In [9]:
# ==============================================================================
# BED GENERATION CONFIGURATION
# ==============================================================================

TOTAL_BEDS = 250

WARD_CSV_PATH = os.path.join(
    "03_Datasets",
    "CSV_Files",
    "dim_wards.csv"
)

OUTPUT_DIRECTORY = os.path.join(
    "03_Datasets",
    "CSV_Files"
)

CSV_FILE_PATH = os.path.join(
    OUTPUT_DIRECTORY,
    "dim_beds.csv"
)

TODAY = datetime.now()


print("=" * 80)
print("BED GENERATION CONFIGURATION")
print("=" * 80)

print(
    f"Total beds : {TOTAL_BEDS}"
)

print(
    f"Ward file  : {WARD_CSV_PATH}"
)

print(
    f"Output path: {CSV_FILE_PATH}"
)

BED GENERATION CONFIGURATION
Total beds : 250
Ward file  : 03_Datasets\CSV_Files\dim_wards.csv
Output path: 03_Datasets\CSV_Files\dim_beds.csv


In [10]:
# ==============================================================================
# LOAD WARD DIMENSION
# ==============================================================================

assert os.path.exists(
    WARD_CSV_PATH
), (
    f"Ward CSV file not found: "
    f"{WARD_CSV_PATH}"
)


wards_df = pd.read_csv(
    WARD_CSV_PATH
)


print("=" * 80)
print("WARD DIMENSION LOADED")
print("=" * 80)

print(
    f"Rows    : {len(wards_df):,}"
)

print(
    f"Columns : {len(wards_df.columns)}"
)

print(
    "\nWard IDs available:"
)

print(
    wards_df[
        "ward_id"
    ].head(10).tolist()
)

WARD DIMENSION LOADED
Rows    : 50
Columns : 14

Ward IDs available:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [11]:
# ==============================================================================
# BED MASTER DATA
# ==============================================================================

BED_TYPES = [
    "Standard Bed",
    "Electric Bed",
    "ICU Bed",
    "Pediatric Bed",
    "Maternity Bed",
    "Bariatric Bed",
    "Emergency Bed"
]


BED_STATUSES = [
    "Active",
    "Inactive",
    "Maintenance"
]


BED_OCCUPANCY_STATUSES = [
    "Occupied",
    "Available",
    "Reserved",
    "Cleaning"
]


EQUIPMENT_LEVELS = [
    "Basic",
    "Standard",
    "Advanced",
    "Critical Care"
]


print("Bed master data loaded successfully.")

Bed master data loaded successfully.


In [12]:
# ==============================================================================
# BED CODE GENERATOR
# ==============================================================================

def generate_bed_code(bed_id):

    return f"BED{bed_id:05d}"


print(
    generate_bed_code(1)
)

print(
    generate_bed_code(TOTAL_BEDS)
)

BED00001
BED00250


In [13]:
# ==============================================================================
# GENERATE BED RECORDS
# ==============================================================================

bed_records = []


ward_ids = wards_df[
    "ward_id"
].tolist()


for bed_id in range(
    1,
    TOTAL_BEDS + 1
):

    # --------------------------------------------------------------------------
    # IDENTIFIERS
    # --------------------------------------------------------------------------

    bed_code = generate_bed_code(
        bed_id
    )


    # --------------------------------------------------------------------------
    # ASSIGN WARD
    # --------------------------------------------------------------------------

    ward_id = random.choice(
        ward_ids
    )


    # --------------------------------------------------------------------------
    # BED NUMBER
    # --------------------------------------------------------------------------

    bed_number = random.randint(
        1,
        50
    )


    # --------------------------------------------------------------------------
    # BED TYPE
    # --------------------------------------------------------------------------

    bed_type = random.choice(
        BED_TYPES
    )


    # --------------------------------------------------------------------------
    # ROOM NUMBER
    # --------------------------------------------------------------------------

    room_number = random.randint(
        101,
        699
    )


    # --------------------------------------------------------------------------
    # FLOOR NUMBER
    # --------------------------------------------------------------------------

    floor_number = random.randint(
        1,
        6
    )


    # --------------------------------------------------------------------------
    # BED STATUS
    # --------------------------------------------------------------------------

    bed_status = random.choices(
        BED_STATUSES,
        weights=[94, 3, 3],
        k=1
    )[0]


    # --------------------------------------------------------------------------
    # OCCUPANCY STATUS
    # --------------------------------------------------------------------------

    if bed_status == "Active":

        bed_occupancy_status = random.choices(
            BED_OCCUPANCY_STATUSES,
            weights=[65, 25, 5, 5],
            k=1
        )[0]

    else:

        bed_occupancy_status = "Available"


    # --------------------------------------------------------------------------
    # ICU BED FLAG
    # --------------------------------------------------------------------------

    if bed_type == "ICU Bed":

        is_icu_bed = True

    else:

        is_icu_bed = False


    # --------------------------------------------------------------------------
    # DAILY BED CHARGE
    # --------------------------------------------------------------------------

    if bed_type == "ICU Bed":

        daily_bed_charge = random.choice(
            [
                5000,
                6000,
                7500,
                8000,
                10000,
                12000
            ]
        )

    elif bed_type in [
        "Maternity Bed",
        "Bariatric Bed"
    ]:

        daily_bed_charge = random.choice(
            [
                2000,
                2500,
                3000,
                3500,
                4000
            ]
        )

    else:

        daily_bed_charge = random.choice(
            [
                500,
                750,
                1000,
                1250,
                1500,
                1750,
                2000
            ]
        )


    # --------------------------------------------------------------------------
    # EQUIPMENT LEVEL
    # --------------------------------------------------------------------------

    if bed_type == "ICU Bed":

        equipment_level = "Critical Care"

    elif bed_type == "Emergency Bed":

        equipment_level = "Advanced"

    else:

        equipment_level = random.choice(
            EQUIPMENT_LEVELS[
                :3
            ]
        )


    # --------------------------------------------------------------------------
    # AUDIT INFORMATION
    # --------------------------------------------------------------------------

    created_date = fake.date_time_between(
        start_date="-5y",
        end_date=TODAY
    )

    last_updated = fake.date_time_between(
        start_date=created_date,
        end_date=TODAY
    )


    # --------------------------------------------------------------------------
    # CREATE RECORD
    # --------------------------------------------------------------------------

    bed_records.append(
        {
            "bed_id": bed_id,
            "bed_code": bed_code,
            "ward_id": ward_id,
            "bed_number": bed_number,
            "bed_type": bed_type,
            "room_number": room_number,
            "floor_number": floor_number,
            "bed_status": bed_status,
            "bed_occupancy_status": bed_occupancy_status,
            "is_icu_bed": is_icu_bed,
            "daily_bed_charge": daily_bed_charge,
            "equipment_level": equipment_level,
            "created_date": created_date,
            "last_updated": last_updated
        }
    )


print(
    f"Generated {len(bed_records):,} "
    f"bed records."
)

Generated 250 bed records.


In [14]:
# ==============================================================================
# CREATE BED DATAFRAME
# ==============================================================================

beds_df = pd.DataFrame(
    bed_records
)


date_columns = [
    "created_date",
    "last_updated"
]


for column in date_columns:

    beds_df[column] = pd.to_datetime(
        beds_df[column]
    )


print("=" * 80)
print("BED DATAFRAME CREATED")
print("=" * 80)

print(
    f"Rows    : {beds_df.shape[0]:,}"
)

print(
    f"Columns : {beds_df.shape[1]}"
)

beds_df.head()

BED DATAFRAME CREATED
Rows    : 250
Columns : 14


,bed_id,bed_code,ward_id,bed_number,bed_type,room_number,floor_number,bed_status,bed_occupancy_status,is_icu_bed,daily_bed_charge,equipment_level,created_date,last_updated
0,1,BED00001,41,8,Standard Bed,382,2,Active,Available,False,1750,Advanced,2022-07-16 14:47:54,2022-08-24 11:16:00
1,2,BED00002,35,6,Maternity Bed,533,1,Active,Occupied,False,4000,Advanced,2023-12-07 05:01:28,2024-12-21 15:15:16
2,3,BED00003,2,36,Electric Bed,659,4,Active,Occupied,False,2000,Basic,2023-06-29 05:09:04,2024-01-31 23:30:07
3,4,BED00004,49,11,Bariatric Bed,533,3,Active,Occupied,False,3000,Basic,2022-06-19 03:56:45,2025-05-05 09:07:52
4,5,BED00005,6,25,Standard Bed,468,3,Active,Available,False,1750,Standard,2026-03-26 02:49:42,2026-04-11 23:59:33


In [15]:
# ==============================================================================
# DATA TYPE CHECK
# ==============================================================================

print("=" * 80)
print("BED DATA TYPES")
print("=" * 80)

print(
    beds_df.dtypes
)

BED DATA TYPES
bed_id                           int64
bed_code                        object
ward_id                          int64
bed_number                       int64
bed_type                        object
room_number                      int64
floor_number                     int64
bed_status                      object
bed_occupancy_status            object
is_icu_bed                        bool
daily_bed_charge                 int64
equipment_level                 object
created_date            datetime64[ns]
last_updated            datetime64[ns]
dtype: object


In [16]:
# ==============================================================================
# BED DATA VALIDATION
# ==============================================================================

print("=" * 80)
print("BED DATA VALIDATION")
print("=" * 80)


# ------------------------------------------------------------------------------
# 1. Row count
# ------------------------------------------------------------------------------

assert (
    len(beds_df)
    == TOTAL_BEDS
)

print(
    "✓ Row count validation passed."
)


# ------------------------------------------------------------------------------
# 2. Bed ID uniqueness
# ------------------------------------------------------------------------------

assert beds_df[
    "bed_id"
].is_unique

print(
    "✓ Bed ID uniqueness validation passed."
)


# ------------------------------------------------------------------------------
# 3. Bed code uniqueness
# ------------------------------------------------------------------------------

assert beds_df[
    "bed_code"
].is_unique

print(
    "✓ Bed code uniqueness validation passed."
)


# ------------------------------------------------------------------------------
# 4. Required fields
# ------------------------------------------------------------------------------

required_columns = [
    "bed_id",
    "bed_code",
    "ward_id",
    "bed_number",
    "bed_type",
    "room_number",
    "floor_number",
    "bed_status",
    "bed_occupancy_status",
    "daily_bed_charge",
    "equipment_level"
]


for column in required_columns:

    assert beds_df[
        column
    ].notna().all(), (
        f"Null values found in {column}"
    )


print(
    "✓ Required field validation passed."
)


# ------------------------------------------------------------------------------
# 5. Bed type validation
# ------------------------------------------------------------------------------

assert beds_df[
    "bed_type"
].isin(
    BED_TYPES
).all()

print(
    "✓ Bed type validation passed."
)


# ------------------------------------------------------------------------------
# 6. Bed status validation
# ------------------------------------------------------------------------------

assert beds_df[
    "bed_status"
].isin(
    BED_STATUSES
).all()

print(
    "✓ Bed status validation passed."
)


# ------------------------------------------------------------------------------
# 7. Occupancy status validation
# ------------------------------------------------------------------------------

assert beds_df[
    "bed_occupancy_status"
].isin(
    BED_OCCUPANCY_STATUSES
).all()

print(
    "✓ Bed occupancy status validation passed."
)


# ------------------------------------------------------------------------------
# 8. Equipment validation
# ------------------------------------------------------------------------------

assert beds_df[
    "equipment_level"
].isin(
    EQUIPMENT_LEVELS
).all()

print(
    "✓ Equipment level validation passed."
)


# ------------------------------------------------------------------------------
# 9. Financial validation
# ------------------------------------------------------------------------------

assert (
    beds_df[
        "daily_bed_charge"
    ] > 0
).all()

print(
    "✓ Daily bed charge validation passed."
)


# ------------------------------------------------------------------------------
# 10. ICU validation
# ------------------------------------------------------------------------------

assert (
    beds_df[
        "is_icu_bed"
    ]
    ==
    beds_df[
        "bed_type"
    ].eq("ICU Bed")
).all()

print(
    "✓ ICU bed validation passed."
)


# ------------------------------------------------------------------------------
# 11. Ward foreign key validation
# ------------------------------------------------------------------------------

assert beds_df[
    "ward_id"
].isin(
    wards_df[
        "ward_id"
    ]
).all()

print(
    "✓ Ward foreign key validation passed."
)


# ------------------------------------------------------------------------------
# 12. Bed code format
# ------------------------------------------------------------------------------

assert beds_df[
    "bed_code"
].str.match(
    r"^BED\d{5}$"
).all()

print(
    "✓ Bed code format validation passed."
)


print()

print(
    "All bed data validations passed successfully. ✓"
)

BED DATA VALIDATION
✓ Row count validation passed.
✓ Bed ID uniqueness validation passed.
✓ Bed code uniqueness validation passed.
✓ Required field validation passed.
✓ Bed type validation passed.
✓ Bed status validation passed.
✓ Bed occupancy status validation passed.
✓ Equipment level validation passed.
✓ Daily bed charge validation passed.
✓ ICU bed validation passed.
✓ Ward foreign key validation passed.
✓ Bed code format validation passed.

All bed data validations passed successfully. ✓


In [17]:
# ==============================================================================
# CHECK DUPLICATE RECORDS
# ==============================================================================

duplicate_count = (
    beds_df
    .duplicated()
    .sum()
)


print("=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)


print(
    f"Duplicate records: {duplicate_count}"
)


assert duplicate_count == 0


print(
    "✓ No duplicate records found."
)

DUPLICATE CHECK
Duplicate records: 0
✓ No duplicate records found.


In [18]:
# ==============================================================================
# WARD FOREIGN KEY VALIDATION
# ==============================================================================

print("=" * 80)
print("WARD FOREIGN KEY CHECK")
print("=" * 80)


valid_ward_ids = set(
    wards_df[
        "ward_id"
    ]
)


invalid_ward_count = (
    ~beds_df[
        "ward_id"
    ].isin(
        valid_ward_ids
    )
).sum()


print(
    f"Invalid ward references: "
    f"{invalid_ward_count}"
)


assert invalid_ward_count == 0


print(
    "✓ All beds reference valid wards."
)

WARD FOREIGN KEY CHECK
Invalid ward references: 0
✓ All beds reference valid wards.


In [19]:
# ==============================================================================
# DATA QUALITY SUMMARY
# ==============================================================================

print("=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)


quality_summary = pd.DataFrame(
    {
        "Metric": [
            "Total Beds",
            "Total Columns",
            "Duplicate Rows",
            "Null Values",
            "Unique Bed IDs",
            "Unique Bed Codes",
            "Unique Ward References",
            "ICU Beds"
        ],

        "Value": [
            len(beds_df),
            len(beds_df.columns),
            beds_df.duplicated().sum(),
            beds_df.isnull().sum().sum(),
            beds_df["bed_id"].nunique(),
            beds_df["bed_code"].nunique(),
            beds_df["ward_id"].nunique(),
            beds_df["is_icu_bed"].sum()
        ]
    }
)


quality_summary

DATA QUALITY SUMMARY


,Metric,Value
0,Total Beds,250
1,Total Columns,14
2,Duplicate Rows,0
3,Null Values,0
4,Unique Bed IDs,250
5,Unique Bed Codes,250
6,Unique Ward References,50
7,ICU Beds,29


In [20]:
# ==============================================================================
# BED SUMMARY
# ==============================================================================

print("=" * 80)
print("BED SUMMARY")
print("=" * 80)


print(
    "\nBed type distribution:"
)

print(
    beds_df[
        "bed_type"
    ].value_counts()
)


print(
    "\nBed status distribution:"
)

print(
    beds_df[
        "bed_status"
    ].value_counts()
)


print(
    "\nBed occupancy status distribution:"
)

print(
    beds_df[
        "bed_occupancy_status"
    ].value_counts()
)


print(
    "\nEquipment level distribution:"
)

print(
    beds_df[
        "equipment_level"
    ].value_counts()
)

BED SUMMARY

Bed type distribution:
bed_type
Standard Bed     44
Emergency Bed    40
Maternity Bed    39
Pediatric Bed    35
Bariatric Bed    33
Electric Bed     30
ICU Bed          29
Name: count, dtype: int64

Bed status distribution:
bed_status
Active         239
Maintenance      6
Inactive         5
Name: count, dtype: int64

Bed occupancy status distribution:
bed_occupancy_status
Occupied     162
Available     65
Cleaning      14
Reserved       9
Name: count, dtype: int64

Equipment level distribution:
equipment_level
Advanced         94
Standard         65
Basic            62
Critical Care    29
Name: count, dtype: int64


In [21]:
# ==============================================================================
# BED FINANCIAL SUMMARY
# ==============================================================================

print("=" * 80)
print("BED FINANCIAL SUMMARY")
print("=" * 80)


financial_summary = beds_df[
    [
        "daily_bed_charge"
    ]
].describe()


financial_summary

BED FINANCIAL SUMMARY


,daily_bed_charge
count,250.000000
mean,2575.000000
std,2479.661243
min,500.000000
25%,1000.000000
50%,1750.000000
75%,3000.000000
max,12000.000000


In [22]:
# ==============================================================================
# EXPORT BED DATA TO CSV
# ==============================================================================

os.makedirs(
    OUTPUT_DIRECTORY,
    exist_ok=True
)


def export_to_csv(beds_df):
    """
    Export bed dimension data into CSV.
    """

    beds_df.to_csv(
        CSV_FILE_PATH,
        index=False
    )


    print("=" * 80)
    print("BED DATA EXPORT")
    print("=" * 80)


    print(
        "CSV file created successfully:"
    )


    print(
        CSV_FILE_PATH
    )


    return CSV_FILE_PATH


csv_file = export_to_csv(
    beds_df
)

BED DATA EXPORT
CSV file created successfully:
03_Datasets\CSV_Files\dim_beds.csv


In [23]:
# ==============================================================================
# VERIFY EXPORTED FILE
# ==============================================================================

assert os.path.exists(
    csv_file
)


file_size_kb = (
    os.path.getsize(csv_file)
    / 1024
)


print("=" * 80)
print("EXPORT VERIFICATION")
print("=" * 80)


print(
    f"File exists : "
    f"{os.path.exists(csv_file)}"
)


print(
    f"File path   : "
    f"{csv_file}"
)


print(
    f"File size   : "
    f"{file_size_kb:.2f} KB"
)

EXPORT VERIFICATION
File exists : True
File path   : 03_Datasets\CSV_Files\dim_beds.csv
File size   : 28.07 KB


In [24]:
# ==============================================================================
# READ BACK EXPORTED CSV
# ==============================================================================

beds_csv_check = pd.read_csv(
    csv_file
)


print("=" * 80)
print("EXPORTED CSV VERIFICATION")
print("=" * 80)


print(
    f"Rows read from CSV    : "
    f"{len(beds_csv_check):,}"
)


print(
    f"Columns read from CSV : "
    f"{len(beds_csv_check.columns)}"
)


assert (
    len(beds_csv_check)
    == TOTAL_BEDS
)


print(
    "✓ Exported CSV row count verified."
)

EXPORTED CSV VERIFICATION
Rows read from CSV    : 250
Columns read from CSV : 14
✓ Exported CSV row count verified.


In [25]:
# ==============================================================================
# DISPLAY SAMPLE BED RECORDS
# ==============================================================================

print("=" * 80)
print("SAMPLE BED RECORDS")
print("=" * 80)


beds_csv_check.head(10)

SAMPLE BED RECORDS


,bed_id,bed_code,ward_id,bed_number,bed_type,room_number,floor_number,bed_status,bed_occupancy_status,is_icu_bed,daily_bed_charge,equipment_level,created_date,last_updated
0,1,BED00001,41,8,Standard Bed,382,2,Active,Available,False,1750,Advanced,2022-07-16 14:47:54,2022-08-24 11:16:00
1,2,BED00002,35,6,Maternity Bed,533,1,Active,Occupied,False,4000,Advanced,2023-12-07 05:01:28,2024-12-21 15:15:16
2,3,BED00003,2,36,Electric Bed,659,4,Active,Occupied,False,2000,Basic,2023-06-29 05:09:04,2024-01-31 23:30:07
3,4,BED00004,49,11,Bariatric Bed,533,3,Active,Occupied,False,3000,Basic,2022-06-19 03:56:45,2025-05-05 09:07:52
4,5,BED00005,6,25,Standard Bed,468,3,Active,Available,False,1750,Standard,2026-03-26 02:49:42,2026-04-11 23:59:33
5,6,BED00006,35,8,Pediatric Bed,181,5,Active,Occupied,False,2000,Standard,2025-03-07 14:07:24,2025-04-01 06:35:43
6,7,BED00007,37,13,Bariatric Bed,172,1,Active,Available,False,2000,Basic,2021-11-05 07:11:37,2022-08-23 09:37:02
7,8,BED00008,7,25,ICU Bed,565,6,Active,Occupied,True,7500,Critical Care,2023-06-15 00:10:51,2024-06-10 10:24:27
8,9,BED00009,14,43,ICU Bed,174,5,Active,Occupied,True,6000,Critical Care,2025-11-21 19:22:24,2026-07-13 14:35:40
9,10,BED00010,11,30,Pediatric Bed,377,6,Active,Occupied,False,1000,Basic,2021-10-26 04:01:02,2026-08-04 21:28:56


In [26]:
# ==============================================================================
# WARD TO BED DISTRIBUTION
# ==============================================================================

print("=" * 80)
print("WARD TO BED DISTRIBUTION")
print("=" * 80)


ward_bed_summary = (
    beds_df
    .groupby(
        "ward_id"
    )
    .agg(
        total_beds=(
            "bed_id",
            "count"
        ),
        icu_beds=(
            "is_icu_bed",
            "sum"
        ),
        average_daily_charge=(
            "daily_bed_charge",
            "mean"
        )
    )
    .reset_index()
)


ward_bed_summary[
    "average_daily_charge"
] = ward_bed_summary[
    "average_daily_charge"
].round(2)


ward_bed_summary.head(20)

WARD TO BED DISTRIBUTION


,ward_id,total_beds,icu_beds,average_daily_charge
0,1,2,1,4000.00
1,2,3,1,4250.00
2,3,3,0,1083.33
3,4,3,0,2333.33
4,5,4,1,2625.00
5,6,3,0,1583.33
6,7,7,2,3535.71
7,8,6,1,3166.67
8,9,7,2,4214.29
9,10,2,0,1250.00


In [27]:
# ==============================================================================
# FINAL BED GENERATION REPORT
# ==============================================================================

print()


print("=" * 80)
print("BED DIMENSION GENERATION COMPLETED")
print("=" * 80)


print(
    f"Total beds              : "
    f"{len(beds_df):,}"
)


print(
    f"Total columns           : "
    f"{len(beds_df.columns)}"
)


print(
    f"Unique wards referenced : "
    f"{beds_df['ward_id'].nunique():,}"
)


print(
    f"ICU beds                : "
    f"{beds_df['is_icu_bed'].sum():,}"
)


print(
    f"Output CSV              : "
    f"{csv_file}"
)


print(
    f"Duplicate records       : "
    f"{beds_df.duplicated().sum()}"
)


print(
    f"Null values             : "
    f"{beds_df.isnull().sum().sum()}"
)


print()


print(
    "Bed dimension generation "
    "completed successfully. ✓"
)


print("=" * 80)


BED DIMENSION GENERATION COMPLETED
Total beds              : 250
Total columns           : 14
Unique wards referenced : 50
ICU beds                : 29
Output CSV              : 03_Datasets\CSV_Files\dim_beds.csv
Duplicate records       : 0
Null values             : 0

Bed dimension generation completed successfully. ✓
